# `bronze_ingest.ipynb` - Capa Bronze (persistencia cruda inmutable)

Persiste, tal cual se han recibido (de la API o del FALLBACK), los datos
de cada ciudad en la capa Bronze, anexando los metadatos minimos de
auditoria tecnica que exige el enunciado: **identificador de lote** y
**marca de tiempo de ingesta en formato UTC**.

> **Decision de diseno (inmutabilidad)**: cada ejecucion escribe un
> fichero Bronze nuevo (`lote_<id>.json`), nunca sobrescribe uno anterior.
> Esto preserva el principio de inmutabilidad de la capa Bronze y, de
> paso, es lo que permite implementar el FALLBACK de
> `extract_openmeteo.ipynb` (leer el ultimo lote real guardado). La
> idempotencia del pipeline (no duplicar registros ni alterar metricas en
> ejecuciones repetidas) se resuelve mas adelante, en la capa Silver, por
> deduplicacion logica sobre `(ciudad, fecha_hora_utc)`; Bronze conserva
> deliberadamente el historial completo de llamadas para trazabilidad.

> **Particionado**: `data/bronze/ciudad=<Nombre>/fecha_ingesta=<AAAA-MM-DD>/lote_<id_lote>.json`,
> por ciudad y por fecha de ingesta.

> Depende de: `config.ipynb`, `logging_config.ipynb`, `extract_openmeteo.ipynb`.

In [ ]:
import os
if os.getcwd().endswith("climate_engine"):
    os.chdir("..")
%run climate_engine/config.ipynb
%run climate_engine/logging_config.ipynb
%run climate_engine/extract_openmeteo.ipynb

In [ ]:
import json
import uuid
from datetime import datetime, timezone
from pathlib import Path

## Funcion principal

In [ ]:
def guardar_bronze(nombre_ciudad, cuerpo_json, origen, id_lote, timestamp_ingesta):
    """Escribe en la capa Bronze el cuerpo JSON recibido de la API (o del
    FALLBACK) para una ciudad, envuelto en un sobre con metadatos minimos
    de auditoria tecnica (id_lote, timestamp_ingesta UTC, origen).

    El fichero es inmutable: nunca se sobrescribe un lote anterior.

    Devuelve: ruta (Path) del fichero Bronze escrito.
    """
    registro = {
        "metadata": {
            "id_lote": id_lote,
            "timestamp_ingesta_utc": timestamp_ingesta,
            "ciudad": nombre_ciudad,
            "origen": origen,
        },
        "datos_api": cuerpo_json,
    }
    fecha_ingesta = timestamp_ingesta[:10]  # AAAA-MM-DD
    directorio = RUTA_BRONZE / f"ciudad={nombre_ciudad}" / f"fecha_ingesta={fecha_ingesta}"
    directorio.mkdir(parents=True, exist_ok=True)
    ruta = directorio / f"lote_{id_lote}.json"
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(registro, f, ensure_ascii=False, indent=2)
    logger.info(f"Bronze escrito para {nombre_ciudad} (origen={origen}): {ruta}")
    return ruta


def ejecutar_ingesta_bronze(ubicaciones=None):
    """Orquesta la extraccion (extract_openmeteo) y la persistencia Bronze
    para el conjunto de ciudades indicado (o todo el catalogo por
    defecto). Todas las ciudades de una misma llamada comparten el mismo
    id_lote, para poder auditar que pertenecen a la misma ejecucion.

    Devuelve: diccionario con el resumen de la ingesta:
        {"id_lote": ..., "rutas": {ciudad: ruta_o_None}, "origenes": {ciudad: origen}}
    """
    id_lote = uuid.uuid4().hex[:12]
    timestamp_ingesta = datetime.now(timezone.utc).isoformat()

    logger.info(f"Inicio de ingesta Bronze. id_lote={id_lote} timestamp_utc={timestamp_ingesta}")
    resultados_extraccion = obtener_datos_todas_las_ciudades(ubicaciones)

    rutas = {}
    origenes = {}
    n_api = n_fallback = n_sin_datos = 0
    for nombre_ciudad, (cuerpo, origen) in resultados_extraccion.items():
        origenes[nombre_ciudad] = origen
        if cuerpo is None:
            rutas[nombre_ciudad] = None
            n_sin_datos += 1
            continue
        rutas[nombre_ciudad] = guardar_bronze(nombre_ciudad, cuerpo, origen, id_lote, timestamp_ingesta)
        if origen == "API":
            n_api += 1
        else:
            n_fallback += 1

    logger.info(
        f"Ingesta Bronze finalizada. id_lote={id_lote} | "
        f"Ciudades desde API: {n_api} | Desde FALLBACK: {n_fallback} | Sin datos: {n_sin_datos}"
    )
    return {"id_lote": id_lote, "rutas": rutas, "origenes": origenes}


def construir_resumen_desde_bronze_existente(ciudades=None):
    """Construye un resumen equivalente al de ejecutar_ingesta_bronze() a
    partir del ultimo lote Bronze YA guardado en disco para cada ciudad,
    sin llamar a la API. Se usa para ejecutar la fase 'silver' de forma
    independiente desde el CLI (seleccion selectiva de fases), sobre lo
    que ya se hubiera ingerido en una ejecucion anterior.

    Devuelve: mismo formato que ejecutar_ingesta_bronze().
    """
    ciudades = ciudades if ciudades is not None else list(UBICACIONES.keys())
    rutas = {}
    origenes = {}
    for nombre_ciudad in ciudades:
        ruta = ruta_ultimo_bronze(nombre_ciudad)
        rutas[nombre_ciudad] = ruta
        origenes[nombre_ciudad] = "BRONZE_EXISTENTE" if ruta else "SIN_DATOS"
    return {"id_lote": None, "rutas": rutas, "origenes": origenes}

## Prueba rapida

Ejecuta la ingesta Bronze real para todas las ciudades del catalogo y muestra el resumen (origen de cada ciudad y ruta del fichero Bronze escrito, cuando lo hay).

In [ ]:
resumen_bronze = ejecutar_ingesta_bronze()

print(f"id_lote de esta ejecucion: {resumen_bronze['id_lote']}")
for ciudad, origen in resumen_bronze["origenes"].items():
    ruta = resumen_bronze["rutas"][ciudad]
    print(f"  {ciudad:12s} | origen={origen:10s} | ruta={ruta}")

n_ok = sum(1 for r in resumen_bronze["rutas"].values() if r is not None)
print(f"\nCiudades con Bronze escrito: {n_ok} / {len(resumen_bronze['rutas'])}")

---
✅ **`bronze_ingest.ipynb` verificado/a.**